# EDA 003.06 — Target Encoding

**Kaggle Feature Engineering Course — Lesson 6**
**Reference:** [Target Encoding](https://www.kaggle.com/code/ryanholbrook/target-encoding) by Ryan Holbrook

---

## Learning Objectives

By the end of this notebook you will understand:

1. What **target encoding** is and how it differs from other categorical encodings
2. Why target encoding can **leak information** and how to prevent it
3. The role of **smoothing** in handling rare categories
4. How to implement **cross-validated target encoding** safely
5. When to choose target encoding vs one-hot vs ordinal encoding

---

## Encoding Categorical Features — A Quick Comparison

| Method | Description | Best for |
|---|---|---|
| **One-hot encoding** | Binary column per category | Low cardinality (< ~15 categories) |
| **Ordinal encoding** | Integer rank (0, 1, 2...) | Ordered categories (e.g. size: S < M < L) |
| **Frequency encoding** | Replace with category count / proportion | High cardinality, no target available |
| **Target encoding** | Replace with mean of target per category | High cardinality, supervised task |

---

## What Is Target Encoding?

> **Target encoding** replaces each category with the **mean of the target variable** among all rows with that category.

**Example:** For a column `city` with target `house_price`:

```
city          → target_encoded_city
New York      → mean(price where city=New York)  = 850,000
Austin        → mean(price where city=Austin)    = 380,000
Chicago       → mean(price where city=Chicago)   = 450,000
```

### The Data Leakage Problem

If you compute the mean using the **same rows you're trying to predict**, the model gets a hint — this is **target leakage**.  
For row $i$ with category $c$, the target mean includes $y_i$ itself.

**Solutions:**
1. **Leave-one-out encoding** — exclude the current row when computing the mean
2. **Cross-fold target encoding** — split data into folds; encode each fold using statistics from the other folds

### The Rare Category Problem

If a category appears only once, its "mean" is just that one row's target — very noisy.

**Smoothing** blends the category mean with the global mean, weighted by count:

$$\hat{y}_c = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}_{global}}{n_c + m}$$

- $n_c$ = number of rows in category $c$
- $\bar{y}_c$ = category mean
- $\bar{y}_{global}$ = overall target mean
- $m$ = smoothing factor (higher → more shrinkage towards global mean)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## Demo 1 — Naive Target Encoding (and Why It Leaks)

In [ ]:
np.random.seed(42)
n = 1000
cities = ['New York', 'Austin', 'Chicago', 'Miami', 'Seattle',
          'Denver', 'Boston', 'Portland', 'Dallas', 'Phoenix']
city_means = {c: v for c, v in zip(cities, [850, 380, 450, 600, 520, 420, 780, 490, 400, 360])}

city_col = np.random.choice(cities, n)
area     = np.random.uniform(50, 250, n)
price    = np.array([city_means[c] for c in city_col]) * (area / 100) + np.random.normal(0, 50, n)

df = pd.DataFrame({'city': city_col, 'area': area, 'price': price})

# Naive target encoding — computed on full dataset (leaky!)
city_target_mean = df.groupby('city')['price'].mean()
df['city_te_naive'] = df['city'].map(city_target_mean)

print("City target means (naive):")
print(city_target_mean.sort_values(ascending=False).round(1))

---
## Demo 2 — Smoothed Target Encoding

In [ ]:
def smooth_target_encode(train_df, col, target, m=10):
    """
    Smoothed target encoding.
    m: smoothing factor — higher = more shrinkage towards global mean
    """
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + m * global_mean) / (stats['count'] + m)
    return smoothed, global_mean

# Simulate a rare city appearing only a few times
df_small = df.copy()
df_small.loc[df_small['city'] == 'Portland', 'city'] = 'Portland'  # only ~100 rows

# Add a very rare city (5 rows)
rare_rows = pd.DataFrame({
    'city': ['TinyTown'] * 5,
    'area': np.random.uniform(50, 150, 5),
    'price': np.random.normal(200, 20, 5)  # only 5 samples, could be noisy
})
df_extended = pd.concat([df_small, rare_rows], ignore_index=True)

naive_means = df_extended.groupby('city')['price'].mean()
smoothed_means, gmean = smooth_target_encode(df_extended, 'city', 'price', m=10)

comparison = pd.DataFrame({'naive': naive_means, 'smoothed': smoothed_means,
                            'count': df_extended.groupby('city')['price'].count()})
comparison['difference'] = (comparison['naive'] - comparison['smoothed']).abs().round(1)
print(f"Global mean: {gmean:.1f}")
comparison.sort_values('count')

**Observe:** `TinyTown` (5 rows, noisy mean) gets smoothed substantially towards the global mean. Common cities with hundreds of rows are barely affected.

---
## Demo 3 — Cross-validated Target Encoding (Leak-free)

The correct approach: for each row, compute the target mean using **only rows in other folds**, never the current row's fold.

In [ ]:
def cv_target_encode(df, col, target, n_splits=5, m=10, random_state=42):
    """
    Cross-validated target encoding — avoids leakage.
    For each fold, the encoding is computed on all OTHER folds.
    """
    encoded = np.full(len(df), np.nan)
    global_mean = df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for train_idx, val_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        val_fold   = df.iloc[val_idx]

        stats = train_fold.groupby(col)[target].agg(['mean', 'count'])
        smoothed = (stats['count'] * stats['mean'] + m * global_mean) / (stats['count'] + m)

        encoded[val_idx] = val_fold[col].map(smoothed).fillna(global_mean).values

    return encoded


df['city_te_cv'] = cv_target_encode(df, 'city', 'price')

# Compare model performance: naive TE vs CV TE vs one-hot
from sklearn.preprocessing import LabelEncoder

df['city_le'] = LabelEncoder().fit_transform(df['city'])

def eval_features(features, df=df):
    scores = cross_val_score(Ridge(), df[features], df['price'], cv=5, scoring='r2')
    return scores.mean()

print("CV R² comparison:")
print(f"  area only                 : {eval_features(['area']):.4f}")
print(f"  area + label encoded city : {eval_features(['area', 'city_le']):.4f}")
print(f"  area + naive TE           : {eval_features(['area', 'city_te_naive']):.4f}")
print(f"  area + CV target encoding : {eval_features(['area', 'city_te_cv']):.4f}")

---
## Demo 4 — Target Encoding for Classification

For binary classification targets (0/1), target encoding replaces each category with the **proportion of positive class** — effectively a probability estimate.

In [ ]:
np.random.seed(0)
products = ['Electronics', 'Clothing', 'Books', 'Home', 'Sports', 'Toys', 'Beauty', 'Food']
churn_rates = [0.15, 0.25, 0.10, 0.20, 0.18, 0.30, 0.12, 0.22]

product = np.random.choice(products, 800)
churn_prob = np.array([churn_rates[products.index(p)] for p in product])
churn = np.random.binomial(1, churn_prob)

df_cls = pd.DataFrame({'product': product, 'churn': churn})
df_cls['product_te'] = cv_target_encode(df_cls, 'product', 'churn')

# Compare to true churn rate
comparison_cls = df_cls.groupby('product').agg(
    true_churn_rate=('churn', 'mean'),
    target_encoded=('product_te', 'mean'),
    count=('churn', 'count')
).round(3)
print(comparison_cls.sort_values('true_churn_rate', ascending=False))

---
## When to Use Target Encoding

| Situation | Recommendation |
|---|---|
| High cardinality categorical (> 15 unique values) | ✅ Strong candidate for target encoding |
| Category has strong relationship with target | ✅ Target encoding captures this directly |
| Small dataset with rare categories | Use smoothing (`m` parameter) |
| Low cardinality | One-hot encoding is simpler and interpretable |
| Need to prevent leakage | Always use cross-validated or hold-out encoding |

---

## Key Takeaways

- Target encoding replaces categories with the **mean of the target** — compact and highly expressive for high-cardinality features
- **Never** compute target encoding statistics on the rows you're predicting — use cross-fold or hold-out encoding
- **Smoothing** prevents overfitting to rare categories by blending with the global mean
- For classification with binary targets, the encoded value is a **probability estimate**

---

## Further Reading

- [Kaggle Tutorial — Target Encoding](https://www.kaggle.com/code/ryanholbrook/target-encoding)
- [sklearn — TargetEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html) (available since sklearn 1.3)
- [Category Encoders library](https://contrib.scikit-learn.org/category_encoders/) — 15+ encoding methods in one package
- [Micci-Barreca (2001) — A preprocessing scheme for high-cardinality categorical attributes](https://dl.acm.org/doi/10.1145/507533.507538) — original smoothed TE paper